# Atividade Prática do Card 8 - Web Scraping com Python p/ Ciência de Dados

Para atividade será relizado o _web scrapping_ de indicadores de uma ação na bolsa. O Site utilizando foi o Status Invest [6].

In [1]:
from bs4 import BeautifulSoup as bs
import requests
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
import time
import pandas as pd

Inicialmente foi utilizado apenas o BeautifulSoup mas alguns elementos não eram renderizados pois exigia trocar de aba no navegador, segue o código comentado de como tinha sido feito inicialmento, pois as instruções utilizadas são úteis para outras páginas e para referência:

In [2]:
'''
url_base = 'https://statusinvest.com.br/acoes/' # URL base para buscar as informações na bolsa.
papel = 'bbas3' # Código do papel do Banco do Brasil na bolsa de valores. A estrutura do site está organizada
                # de uma maneira em que o papel do ativo identifica a página do site.

url = url_base + papel

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7"
} # Cabeçalho utilizado para imitar um navegador, pois quando eu tentava acessar o Cloudfare negava o acesso para 
    # prevenir ataques ao site. Para os valores utilizei as mesmas informações do meu navegador ao acessar o site pelo 
    # Google Chrome. 

response = requests.get(url, headers=headers) # Header aplicados conforme [7].

with open(f'{papel}.html', 'w', encoding='utf-8') as arquivo: # Salvei o conteúdo do HTML em um arquivo de backup
                                                                # caso não esteja acessível o site.
    arquivo.write(response.text)
'''
print()

Foi utilizado o Selenium para iteragir com a página e buscar as informações para conesguir utilizar o Beautiful Soup com o HTML necessário, segue o código fonte: 

In [3]:

url_base = 'https://statusinvest.com.br/acoes/' # URL base para buscar as informações na bolsa.
papel = 'bbas3' # Código do papel do Banco do Brasil na bolsa de valores. A estrutura do site está organizada
                # de uma maneira em que o papel do ativo identifica a página do site.

url = url_base + papel

# Utilizado Selenium conforme demonstrado em: para instalação [8], para utilizar o navegador [9]
# e para entender como selecionar e navegar nos elementos da página [10].
driver = webdriver.Chrome()

driver.get(url)

print(driver.title) # Confere se conseguiu pegar a página.

# driver.implicitly_wait(10) # Espera 10 segundos até que o nagevador busque pelo
                            # elemento [12].

botao_historico = driver.find_element(By.CSS_SELECTOR, 'button[title="Histórico do ativo"]') # Busca pelo botão com a 
                                                                                                # tabela que preciso. [11]

wait = WebDriverWait(driver, timeout=2)
wait.until(lambda _: botao_historico.is_displayed()) # Espera o botão estar rendereziado 
                                                        # conforme específica para que o 
                                                        # clique na aba possa acontecer [12].

wait = WebDriverWait(driver, timeout=2)
wait.until(lambda _: botao_historico.is_displayed()) # Espera o botão estar rendereziado 
                                                        # conforme específica [12].

ActionChains(driver).move_to_element(botao_historico).perform() # Movimenta a primeria vez para chegar na página e carregar 
                                                                # novamente no navegador [13] e [14].

ActionChains(driver).move_to_element_with_offset(botao_historico, 8, 0).perform() # Como estava dando erro ao utilizar
                                                                                    # apenas o click (não renderizava a 
                                                                                    # tabela e a ação de clique não 
                                                                                    # acontecia no site) foi simulado o 
                                                                                    # movimento do mouse até o botão da 
                                                                                    # página conforme [13].
                                                                                    # Como pode acontecer do conteúdo não
                                                                                    # estar na tela do navegador ou algum 
                                                                                    # anúncio de rodapé interceptar o clique
                                                                                    # foi utilizado o movimentar com `offset`.
                                                                                    # conforme [14].

botao_historico.click() # Clica no botão selecionado para que mude de aba [10].

time.sleep(5) # Esperar a página carregar para poder acessar o html dela.

html_pagina = driver.page_source # Acessa o código fonte da página.

pagina = bs(html_pagina, 'html.parser') # Ao invés de fazer o parse pela URL fazemos o parse pelo
                                        # próprio HTML conforme [2].

print(pagina.head.title) # Conferindo se carregou de acordo.

with open(f'selenium_{papel}.html', 'w', encoding='utf-8') as arquivo: # Salvei o conteúdo do HTML em um arquivo de backup
                                                                # caso não esteja acessível o site.
    arquivo.write(html_pagina)

print('fim da execução') # Para saber que terminou OK

BBAS3 - BANCO BRASIL ON: cotação e indicadores
<title>BBAS3 - BANCO BRASIL ON: cotação e indicadores</title>
fim da execução


In [4]:
time.sleep(5) # Espera para poder conferir e manter o navegador renderizadoa até
                # que os dados sejam carregados.
driver.quit() # Fecha o navegador do Selenium

In [5]:
# pagina = bs(response.text) # Montando o objeto BeautifulSoup para começar o _web scraping_

pagina.head.title # Conferindo se carregou de acordo.

<title>BBAS3 - BANCO BRASIL ON: cotação e indicadores</title>

In [6]:
'''
tabela_indicadores = pagina.select(
    'div.indicator-historical-container div.indicators'
)  # Acessa as tabelas de indicadores do ativo, sao cinco seções de indicadores por isso uma lista com 5 itens
   
print(len(tabela_indicadores), '\n')

secao_indicadores = tabela_indicadores[0] # Com essa lista podemos percorrê-la e trazer os respectivos indicadores.
# print(secao_indicadores.prettify())
'''
print()

In [7]:
tabela_indicadores = pagina.select(
    'div.indicator-historical-container div.indicators'
)  # Acessa as tabelas de indicadores do ativo, sao cinco seções de indicadores por isso uma lista com 5 itens
   
print(len(tabela_indicadores), '\n')

print(tabela_indicadores[0].find('h3').prettify()) # Mostrando para ficar mais fácil de visualizar
                                                    # a árvore de elemtos da página

indices_dataframe = []
for indice_indicadores in tabela_indicadores: # Com essa lista podemos percorrê-la e trazer os respectivos indicadores.
    descricao_indices = [h3.get_text() for h3 in indice_indicadores.find_all('h3')]
    indices_dataframe.extend(descricao_indices)
print(len(indices_dataframe))
print(indices_dataframe)

print(type(indices_dataframe[0]))

5 

<h3 class="title m-0 uppercase">
 D.Y
</h3>

30
['D.Y', 'P/L', 'PEG Ratio', 'P/VP', 'EV/EBITDA', 'EV/EBIT', 'P/EBITDA', 'P/EBIT', 'VPA', 'P/Ativo', 'LPA', 'P/SR', 'P/Cap. Giro', 'P/Ativo Circ. Liq.', 'Dív. líquida/PL', 'Dív. líquida/EBITDA', 'Dív. líquida/EBIT', 'PL/Ativos', 'Passivos/Ativos', 'Liq. corrente', 'M. Bruta', 'M. EBITDA', 'M. EBIT', 'M. Líquida', 'ROE', 'ROA', 'ROIC', 'Giro ativos', 'CAGR Receitas 5 anos', 'CAGR Lucros 5 anos']
<class 'str'>


In [8]:
colunas_indicadores = pagina.select(
    'div.indicator-historical-container div.th'
)

print(len(colunas_indicadores)) # Como o tamanho é 50 pois há 5 seções com cabecalho. Vamos pegar 
                                # apenas a primeira seção


colunas_indicadores = pagina.select(
    'div.indicator-historical-container div[data-group="0"] div.th' 
)
print(len(colunas_indicadores)) # Agora temos apenas as 10 colunas, o que é suficente já que ela 
                                # se repete.

print(colunas_indicadores[0].prettify())

colunas_dataframe = [coluna.get_text() for coluna in colunas_indicadores] # Usando o `get_text` 
                                                                            # ao invés de `string` 
                                                                            # pois evita casos `None`
                                                                            # No caso não ocorre 
                                                                            # mas como o retorno é 
                                                                            # o mesmo foi preferido 
                                                                            # utilizar.
print(colunas_dataframe)

50
10
<div class="th w-100 timeType-1">
 ATUAL
</div>

['ATUAL', '2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017']


In [9]:
valores_indicadores = pagina.select(
    'div.indicator-historical-container div.td'
)

print(len(valores_indicadores)) # Como são 30 indíces e 10 colunas a quantidade de elementos é 300.



print(valores_indicadores[0].prettify())
print(valores_indicadores[1].prettify())
print(valores_indicadores[2].prettify())

dados = {} # Dicionário para organizar os dados das tabelas

col = 0 # Para iterar no sentido das colunas.
row = 0 # Para iterar no sentido das linhas (indíces)
for valor_div in valores_indicadores:
    valor_text = valor_div.get_text().strip() # `strip` para evitar espaços na comparação

    if '%' in valor_text and '%' not in indices_dataframe[row]: # Altera o indíce adicionando o 
                                                                # % quando o valor indica percentual
        indices_dataframe[row] = indices_dataframe[row]+"%"
    
    valor = None
    if valor_text not in ['-', '-%']: # Se verdadeiro possui valor numérico 
        
        valor_str = valor_text.replace('.', '') # Trata os milhares
        valor_str = valor_str.replace(',','.') # Trata para depois converter em `float`
        valor_str = valor_str.replace('%','') # Remove as porcentagens para converter em `float`
        valor = float(valor_str)

    coluna_atual = colunas_dataframe[col]
    if coluna_atual not in dados: # Caso não tenha a chave no dicionário no momento.
        dados[coluna_atual] = [valor] # Inicia a lista de valores para a coluna.
    else: 
        dados[coluna_atual].append(valor) # Adiciona o valor na lista

    # Como os dados na listam representam o HTML da direita para esquerda e pulam para uma nova
    # linha, a cada iteração que chega a quantidade de colunas se inicia uma nova linha.
    if col == len(colunas_dataframe) - 1:
        col = 0
        row += 1 # A cada coluna que termina de percorrer começa a percorrer uma nova linha.
    else:
        col += 1

    

300
<div class="td w-100 timeType-1">
 2,58%
</div>

<div class="td w-100 timeType-0">
 5,40%
</div>

<div class="td w-100 timeType-0">
 13,42%
</div>



In [10]:
df = pd.DataFrame(dados, index=indices_dataframe) # Inicializa o DataFrame com os dados obtidos.
df.head(30)

,ATUAL,2025,2024,2023,2022,2021,2020,2019,2018,2017
D.Y%,2.58,5.40,13.42,8.48,12.01,7.86,3.81,4.84,3.26,2.96
P/L,7.79,7.49,4.75,4.66,3.32,4.19,8.36,9.23,9.64,8.58
PEG Ratio,-0.21,-0.18,NaN,NaN,NaN,NaN,NaN,NaN,0.32,0.17
P/VP,0.64,0.66,0.77,0.91,0.62,0.58,0.95,1.53,1.45,1.04
EV/EBITDA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.01,NaN,NaN
EV/EBIT,19.25,22.38,5.00,3.76,2.59,3.45,9.60,13.59,6.53,5.72
P/EBITDA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.01,NaN,NaN
P/EBIT,19.25,22.38,5.00,3.76,2.59,3.45,9.60,13.59,6.53,5.72
VPA,33.45,33.02,31.34,59.06,55.58,49.82,40.74,34.51,32.10,30.55
P/Ativo,0.05,0.05,0.06,0.07,0.05,0.04,0.06,0.10,0.09,0.06


In [11]:
import datetime

data_hora = str(datetime.datetime.now())
data_hora = data_hora[0:-7]
data_hora = data_hora.replace(' ', '__')
data_hora = data_hora.replace(':', '-')

file_name = f'indicadores_{papel}_{data_hora}.csv' 

df.to_csv(file_name, sep=';', decimal='.') # Salvando o arquivo com a data hora da análise